## tl;dr

A read-only snapshot of the four active Agent Studio projects found no ready or running tasks, while 426 tasks were in review, 391 were blocked, one was waiting, and 389 interventions were open. The market scan therefore prioritizes artifact lineage, review surfaces, versioned runbooks/evals, and outcome measurement ahead of adding more generic agents.

## Context & Methods

This is a companion analysis notebook for an executive product-strategy report. It combines a read-only SQLite snapshot of Lucas Agent Studio with a structured review of 18 AI companies and startups current through August 22, 2026.

### Key Assumptions

- Task and intervention counts are operational state, not measures of business value or agent quality.
- Company feature claims are taken primarily from official product documentation; they establish product design patterns, not independently audited outcomes.
- Roadmap scores are structured expert judgment on a 0-100 scale, not empirical performance estimates.

In [1]:
from pathlib import Path
import json
import sqlite3

RUNTIME_DB = Path(r"C:\\AI\\projects\\LucasAgentStudio\\data\\company-runtime\\company.sqlite")
WEIGHTS = {"pain_now": 0.30, "cross_project_leverage": 0.25, "architecture_fit": 0.20, "trust_defensibility": 0.15, "delivery_ease": 0.10}
assert RUNTIME_DB.exists(), RUNTIME_DB
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9
print(RUNTIME_DB)

C:\AI\projects\LucasAgentStudio\data\company-runtime\company.sqlite


## Data

### 1. Read the active runtime frontier

In [2]:
task_query = """
WITH statuses(status) AS (
  VALUES ('ready'), ('running'), ('review'), ('blocked'), ('waiting')
)
SELECT p.name AS project, s.status, COUNT(t.id) AS task_count
FROM projects AS p
CROSS JOIN statuses AS s
LEFT JOIN tasks AS t ON t.project_id = p.id AND t.status = s.status
WHERE p.desired = 1 AND p.archived_at IS NULL
GROUP BY p.name, s.status
ORDER BY p.name, s.status
"""
intervention_query = """
SELECT COUNT(*) AS open_interventions
FROM interventions AS i
JOIN projects AS p ON p.id = i.project_id
WHERE p.desired = 1 AND p.archived_at IS NULL
  AND i.status = 'open' AND i.archived_at IS NULL
"""
connection = sqlite3.connect(f"file:{RUNTIME_DB.as_posix()}?mode=ro", uri=True)
connection.row_factory = sqlite3.Row
active_task_rows = [dict(row) for row in connection.execute(task_query)]
open_interventions = connection.execute(intervention_query).fetchone()[0]
connection.close()
print(json.dumps({"active_task_rows": active_task_rows, "open_interventions": open_interventions}, indent=2))

{
  "active_task_rows": [
    {
      "project": "AnimeRPG \u00b7 Astral Engine",
      "status": "blocked",
      "task_count": 173
    },
    {
      "project": "AnimeRPG \u00b7 Astral Engine",
      "status": "ready",
      "task_count": 0
    },
    {
      "project": "AnimeRPG \u00b7 Astral Engine",
      "status": "review",
      "task_count": 177
    },
    {
      "project": "AnimeRPG \u00b7 Astral Engine",
      "status": "running",
      "task_count": 0
    },
    {
      "project": "AnimeRPG \u00b7 Astral Engine",
      "status": "waiting",
      "task_count": 1
    },
    {
      "project": "Every Aircraft",
      "status": "blocked",
      "task_count": 147
    },
    {
      "project": "Every Aircraft",
      "status": "ready",
      "task_count": 0
    },
    {
      "project": "Every Aircraft",
      "status": "review",
      "task_count": 149
    },
    {
      "project": "Every Aircraft",
      "status": "running",
      "task_count": 0
    },
    {
      "project": "

### 2. Define the roadmap decision model

In [3]:
priority_inputs = [
    {"pattern": "Artifact graph + explicit promotion", "pain_now": 5, "cross_project_leverage": 5, "architecture_fit": 5, "trust_defensibility": 5, "delivery_ease": 3},
    {"pattern": "Versioned runbooks + eval gates", "pain_now": 5, "cross_project_leverage": 5, "architecture_fit": 5, "trust_defensibility": 5, "delivery_ease": 2},
    {"pattern": "Native review workbench", "pain_now": 5, "cross_project_leverage": 5, "architecture_fit": 4, "trust_defensibility": 5, "delivery_ease": 3},
    {"pattern": "Outcome ledger", "pain_now": 4, "cross_project_leverage": 5, "architecture_fit": 5, "trust_defensibility": 5, "delivery_ease": 4},
    {"pattern": "Permissioned project context graph", "pain_now": 4, "cross_project_leverage": 5, "architecture_fit": 4, "trust_defensibility": 5, "delivery_ease": 3},
    {"pattern": "Task-aware model routing + health", "pain_now": 4, "cross_project_leverage": 5, "architecture_fit": 4, "trust_defensibility": 4, "delivery_ease": 2},
    {"pattern": "Signal radar + action packs", "pain_now": 4, "cross_project_leverage": 3, "architecture_fit": 5, "trust_defensibility": 4, "delivery_ease": 4},
    {"pattern": "Long-horizon proactive agents", "pain_now": 3, "cross_project_leverage": 4, "architecture_fit": 5, "trust_defensibility": 3, "delivery_ease": 2},
    {"pattern": "Publish/export productization", "pain_now": 3, "cross_project_leverage": 5, "architecture_fit": 3, "trust_defensibility": 3, "delivery_ease": 3},
    {"pattern": "Generic agent marketplace", "pain_now": 1, "cross_project_leverage": 3, "architecture_fit": 3, "trust_defensibility": 1, "delivery_ease": 2},
]
priority_scores = []
for row in priority_inputs:
    weighted = sum(row[criterion] * weight for criterion, weight in WEIGHTS.items())
    priority_scores.append({**row, "priority_score": round(weighted / 5 * 100)})
priority_scores.sort(key=lambda row: (-row["priority_score"], row["pattern"]))
print(json.dumps(priority_scores, indent=2))

[
  {
    "pattern": "Artifact graph + explicit promotion",
    "pain_now": 5,
    "cross_project_leverage": 5,
    "architecture_fit": 5,
    "trust_defensibility": 5,
    "delivery_ease": 3,
    "priority_score": 96
  },
  {
    "pattern": "Versioned runbooks + eval gates",
    "pain_now": 5,
    "cross_project_leverage": 5,
    "architecture_fit": 5,
    "trust_defensibility": 5,
    "delivery_ease": 2,
    "priority_score": 94
  },
  {
    "pattern": "Native review workbench",
    "pain_now": 5,
    "cross_project_leverage": 5,
    "architecture_fit": 4,
    "trust_defensibility": 5,
    "delivery_ease": 3,
    "priority_score": 92
  },
  {
    "pattern": "Outcome ledger",
    "pain_now": 4,
    "cross_project_leverage": 5,
    "architecture_fit": 5,
    "trust_defensibility": 5,
    "delivery_ease": 4,
    "priority_score": 92
  },
  {
    "pattern": "Permissioned project context graph",
    "pain_now": 4,
    "cross_project_leverage": 5,
    "architecture_fit": 4,
    "trust_defe

### 3. Preserve the company/source inventory

In [4]:
company_sources = [
    ("JoinRunway", "https://www.joinrunway.io/"),
    ("Cursor", "https://cursor.com/blog/agent-best-practices"),
    ("Devin", "https://docs.devin.ai/get-started/devin-intro"),
    ("Replit", "https://docs.replit.com/learn/build-with-agent"),
    ("LangChain / LangGraph / LangSmith", "https://www.langchain.com/langsmith-platform"),
    ("CrewAI", "https://docs.crewai.com/"),
    ("Sierra", "https://sierra.ai/product"),
    ("Decagon", "https://decagon.ai/product/aop"),
    ("Glean", "https://www.glean.com/blog/agent-dev-lifecycle-2026"),
    ("Harvey", "https://www.harvey.ai/platform/agents"),
    ("Hebbia", "https://www.hebbia.com/product"),
    ("Abridge", "https://www.abridge.com/product"),
    ("RunwayML", "https://help.runwayml.com/hc/en-us/articles/53718574533395-Viewing-and-downloading-an-asset-s-lineage"),
    ("ElevenLabs", "https://elevenlabs.io/docs/eleven-creative/products/studio"),
    ("Perplexity", "https://www.perplexity.ai/help-center/en/articles/10352961-what-are-spaces"),
    ("Granola", "https://www.granola.ai/blog/two-dot-zero"),
    ("Gamma", "https://gamma.app/"),
    ("Suno", "https://about.suno.com/blog/studio-2"),
]
assert len(company_sources) == len({name for name, _ in company_sources}) == 18
assert all(url.startswith("https://") for _, url in company_sources)
print(f"Reviewed {len(company_sources)} companies/startups with one primary-source anchor each.")

Reviewed 18 companies/startups with one primary-source anchor each.


## Results

### Validate the headline quantities

In [5]:
projects = sorted({row["project"] for row in active_task_rows})
status_totals = {}
for row in active_task_rows:
    status_totals[row["status"]] = status_totals.get(row["status"], 0) + row["task_count"]
nonterminal_queue = sum(status_totals.get(status, 0) for status in ("review", "blocked", "waiting"))
assert len(projects) == 4
assert len(active_task_rows) == 20
assert all(row["task_count"] >= 0 for row in active_task_rows)
assert all(0 <= row["priority_score"] <= 100 for row in priority_scores)
headline = {
    "active_projects": len(projects),
    "ready_plus_running": status_totals.get("ready", 0) + status_totals.get("running", 0),
    "review": status_totals.get("review", 0),
    "blocked": status_totals.get("blocked", 0),
    "waiting": status_totals.get("waiting", 0),
    "nonterminal_queue": nonterminal_queue,
    "open_interventions": open_interventions,
    "top_priorities": [{"pattern": row["pattern"], "score": row["priority_score"]} for row in priority_scores[:5]],
}
print(json.dumps(headline, indent=2))

{
  "active_projects": 4,
  "ready_plus_running": 0,
  "review": 426,
  "blocked": 391,
  "waiting": 1,
  "nonterminal_queue": 818,
  "open_interventions": 389,
  "top_priorities": [
    {
      "pattern": "Artifact graph + explicit promotion",
      "score": 96
    },
    {
      "pattern": "Versioned runbooks + eval gates",
      "score": 94
    },
    {
      "pattern": "Native review workbench",
      "score": 92
    },
    {
      "pattern": "Outcome ledger",
      "score": 92
    },
    {
      "pattern": "Permissioned project context graph",
      "score": 86
    }
  ]
}


## Takeaways

1. The current constraint is turning accumulated work into trusted, promoted artifacts; adding agent count would increase review debt.
2. The strongest companies converge on persistent context, bounded plans, versioned execution, native evidence review, explicit promotion, and outcome feedback.
3. The recommended positioning is an evidence-driven Project Operating System with vertical workflow packs, not a generic marketplace of agent personas.